# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_SESSION_1_structured_spec_langgraph.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About this Notebook

Structured specs + the JSON trap + validated LangGraph handoffs

Two messy stakeholder inputs become **two different contracts**:

1. **Agent workflow spec** — capabilities, guardrails, escalation, tools, HITL.
2. **Coding scaffold** — language, inputs, processing steps, outputs, dependencies, edge cases.

This notebook goes heavy on **Pydantic v2** and adds a **minimal LangGraph** where the critical idea is:

> **Structured output from an LLM is still untrusted bytes until you validate it again at the handoff boundary.**

That is the **JSON trap**: `json.loads` succeeds, keys look right, and you still do not have a contract-safe object until `model_validate` / `model_validate_json` passes.

**How to use this notebook:** Run cells top to bottom with `OPENAI_API_KEY` set. Each numbered section introduces one idea; code cells stay small so you can rerun after you change schemas or prompts.

**Roadmap (what each section is for)**

| Section | What it does | Why it matters |
|--------|----------------|----------------|
| **1 — WorkflowSpec** | Defines the *agent-shaped* contract (nested escalation + tools + HITL rules). | Downstream routers, policy checks, and graph builders need **typed slots**, not a paragraph of prose. |
| **2 — CodingScaffold** | Defines the *codegen-shaped* contract (ordered steps, I/O, edge cases). | IDEs, codegen, and CI can consume **pipelines**; a flat `details[]` list does not tell you order or dependencies. |
| **3 — JSON trap** | Shows JSON that parses vs. payloads that **validate**. | Teams stop trusting “valid JSON” as if it were a domain object. |
| **4 — Extraction** | LLM extracts `WorkflowSpec` / `CodingScaffold` via `with_structured_output`. | Fast iteration in notebooks; still **not** a substitute for boundary checks (see below). |
| **5 — LangGraph** | Classify → extract → serialize → **re-validate** → emit or fail. | Makes **every handoff** an explicit, testable step—closer to how you should ship agents. |
| **6 — Corrupt handoff** | Mutates **nested** JSON (a single `step_id`) after “the wire”. | Models “real” bugs: partial sanitizers, UI labels, bad merges—not only missing top-level keys. |
| **7 — Notes** | Extensions (JSON Schema, repair loops). | Where to take the class next.






- **Nested models** — A flat JSON blob cannot tell an *escalation* from a *capability*. Nested types encode **roles**, so the next node does not re-interpret the same string three different ways.
- **Validators + constraints** — Catch silent garbage (slug rules, duplicate step ids, “escalation but no HITL”) **before** tickets, graphs, or codegen consume the payload. Fail fast at the boundary you control.
- **Wire format ≠ domain model** — Over HTTP, queues, or a database row you only have bytes. **Re-validation on receive** is how one team’s “obvious” JSON becomes another team’s **guaranteed** `WorkflowSpec` (or a hard error).
- **LangGraph as pedagogy** — Each node is a *process boundary*. If validation is not in the graph, it is usually **implicit and missing** in production.
- **Corrupt on purpose** — Proves that even perfect extraction earlier in the pipeline does not protect you after an intermediate service touches the payload.


In [ ]:
!pip install -q "langgraph>=0.2" "langchain-openai>=0.3,<1.0" "python-dotenv>=1.0,<2.0" "pydantic>=2.7,<3"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 1.7 MB/s eta 0:00:00


In [ ]:
%env NEBIUS_API_KEY="v1.CmQKHHN0YXRpY2tleS1lMDBlaDB4ZnhjOTJnM2FwaDESIXNlcnZpY2VhY2NvdW50LWUwMGg2Z21kNmszcWgycXRycTIMCNWdmc4GEPrCsdEBOgwI1KCxmQcQwKP10ANAAloDZTAw.AAAAAAAAAAGmLbk7vAfDFQT8zojCUKv50GQWHV5cqqdZW6MU8vPQyhpazDF___-nSCtFwJPgcvc2CPP1H6KoUO6Nb5jfcboH"

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()

NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=NEBIUS_API_KEY,
    base_url="https://api.tokenfactory.nebius.com/v1/",
    temperature=0,
)

resp = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print(resp.content)

Hello, I hope you’re having a wonderful day!


In [ ]:
from __future__ import annotations

import json
import os
import re
from typing import Any, Literal, Self, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    TypeAdapter,
    ValidationError,
    computed_field,
    field_validator,
    model_validator,
)

load_dotenv()
#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


#llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)




False

## Environment setup (dependencies + imports)

The **pip** cell installs:

- **`pydantic`** — schemas, validators, JSON parsing into typed models (`model_validate_json`).
- **`langchain-openai` + `langgraph`** — `ChatOpenAI` + `with_structured_output` and a tiny state machine.

The **imports** cell wires the LLM (temperature 0 for stable teaching runs), shared regex helpers, and LangGraph’s `StateGraph` / `START` / `END`.

**Why separate?** In class you can swap models or pin versions without hunting through model definitions.


## 1) Pydantic: agent **WorkflowSpec** (nested + invariants)

**What this section does:** Defines the **smallest useful contract** for describing an internal assistant: what it may do, what it must refuse, when to escalate, which tools it needs, and whether humans belong in the loop.

**Why it matters:** Product and policy people write prose. Your **router, tool registry, and HITL subgraph** need *fields*. If those fields are vague, every downstream step re-LLMs the same paragraph and you have no single source of truth.

**Pieces in the code cell**

- **`EscalationTrigger`** — One record per “when this happens, route to X”. Forces `target_team` and `severity` instead of a free-text blob.
- **`ToolNeed`** — Integrations are not just names; each has a **purpose** so codegen or platform teams know *why* the tool exists.
- **`@field_validator` on tool `name`** — Enforces **slug-safe** identifiers (`vector_db`, not `"Vector DB "`). Downstream labels, env vars, and metric tags stay stable.
- **`@model_validator` (escalations ⇒ HITL)** — A **business invariant**: you cannot claim automated escalation to legal without admitting humans in the loop. Catches inconsistent specs before the graph is built.
- **`computed_field` `escalation_count`** — Example of derived data you can expose on the model (and in `model_dump`) without duplicating source fields.

- **`ConfigDict(str_strip_whitespace=True, validate_assignment=True)`** — Trims sloppy strings and re-validates when you mutate attributes—closer to production objects than one-shot dicts.


In [ ]:
_SLUG = re.compile(r'^[a-z][a-z0-9_]{1,63}$')


class EscalationTrigger(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    condition: str = Field(
        min_length=4,
        max_length=240,
        description="When to escalate (natural language, one clause).",
    )
    target_team: Literal["legal", "security", "support", "management", "other"] = Field(
        description="Who receives the escalation."
    )
    severity: Literal["low", "medium", "high"] = "medium"


class ToolNeed(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    name: str = Field(description="Machine-friendly tool or integration id.")
    purpose: str = Field(min_length=8, max_length=400)
    required: bool = True

    @field_validator("name", mode="after")
    @classmethod
    def slug_name(cls, v: str) -> str:
        raw = v.strip().lower().replace(" ", "_").replace("-", "_")
        if not _SLUG.match(raw):
            raise ValueError(f"tool name must match {_SLUG.pattern}, got {v!r}")
        return raw


class WorkflowSpec(BaseModel):
    '''Contract for building router + tool nodes + HITL in LangGraph (or similar).'''

    model_config = ConfigDict(str_strip_whitespace=True, validate_assignment=True)

    system_goal: str = Field(min_length=8, max_length=500)
    capabilities: list[str] = Field(
        min_length=1,
        max_length=16,
        description="What the assistant is allowed to do end-to-end.",
    )
    restricted_actions: list[str] = Field(
        default_factory=list,
        max_length=16,
        description="Actions or topics the assistant must refuse or block.",
    )
    escalation_triggers: list[EscalationTrigger] = Field(default_factory=list, max_length=12)
    required_tools: list[ToolNeed] = Field(default_factory=list, max_length=16)
    human_in_loop: bool = Field(description="True if humans must approve or take over some paths.")
    open_questions: list[str] = Field(default_factory=list, max_length=20)

    @field_validator("capabilities", mode="after")
    @classmethod
    def strip_capabilities(cls, v: list[str]) -> list[str]:
        out = [s.strip() for s in v if s and str(s).strip()]
        if not out:
            raise ValueError("capabilities must contain at least one non-empty string")
        return out

    @field_validator("restricted_actions", mode="after")
    @classmethod
    def strip_restricted(cls, v: list[str]) -> list[str]:
        return [s.strip() for s in v if s and str(s).strip()]

    @model_validator(mode="after")
    def escalations_require_hitl(self) -> Self:
        if self.escalation_triggers and not self.human_in_loop:
            raise ValueError("escalation_triggers is non-empty but human_in_loop is False")
        return self

    @computed_field  # type: ignore[prop-decorator]
    @property
    def escalation_count(self) -> int:
        return len(self.escalation_triggers)


## 2) Pydantic: **CodingScaffold** (steps as models + computed field)

**What this section does:** Turns “build a small script that…” into an **ordered pipeline**: typed steps, explicit I/O, optional dependencies and edge cases.

**Why it matters:** Code generators, README templates, and review bots need **structure**—not a single block of text. Ordered `ProcessingStep` items are a minimal executable plan; they map naturally to functions, tickets, or test cases.

**Pieces in the code cell**

- **`ProcessingStep`** — Each step has a **stable `step_id`** and a human-readable `description`. The graph of work is explicit.
- **`@field_validator` on `step_id`** — Same slug discipline as tools: stable identifiers across exports.
- **`@model_validator` (unique `step_id`s)** — Prevents ambiguous pipelines before you generate code.
- **`computed_field` `suggested_entrypoint`** — Shows how to attach **conventions** (e.g. `main.py::main`) without storing redundant columns in every payload.


In [ ]:
class ProcessingStep(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    step_id: str = Field(description="snake_case id, e.g. read_csv")
    description: str = Field(min_length=6, max_length=300)

    @field_validator("step_id", mode="after")
    @classmethod
    def slug_step(cls, v: str) -> str:
        raw = v.strip().lower().replace(" ", "_").replace("-", "_")
        if not _SLUG.match(raw):
            raise ValueError(f"step_id must match slug pattern, got {v!r}")
        return raw


class CodingScaffold(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, validate_assignment=True)

    language: Literal["python", "typescript", "go", "other"] = "python"
    input_type: str = Field(min_length=2, max_length=120)
    processing_steps: list[ProcessingStep] = Field(min_length=1, max_length=24)
    output_type: str = Field(min_length=2, max_length=120)
    dependencies: list[str] = Field(default_factory=list, max_length=32)
    edge_cases: list[str] = Field(default_factory=list, max_length=24)
    open_questions: list[str] = Field(default_factory=list, max_length=20)

    @computed_field  # type: ignore[prop-decorator]
    @property
    def suggested_entrypoint(self) -> str:
        lang = self.language
        if lang == "python":
            return "main.py::main"
        if lang == "typescript":
            return "src/index.ts::main"
        return "README + single entry script"

    @model_validator(mode="after")
    def steps_order_unique(self) -> Self:
        ids = [s.step_id for s in self.processing_steps]
        if len(ids) != len(set(ids)):
            raise ValueError(f"duplicate step_id in processing_steps: {ids}")
        return self


## 3) The JSON trap (hand-off / wire format)

**What this section does:** Separates **“parses as JSON”** from **“is a valid `WorkflowSpec` / union member”**.

**Why it matters:** Logs, queues, and browser clients give you strings. `json.loads` only guarantees syntax. **Semantics** (required fields, correct types, nested shape) and **policies** (e.g. escalations require HITL) live in Pydantic (or JSON Schema, protobuf, etc.).

**The two code cells**

1. **First cell** — JSON that **loads** but violates a **cross-field rule** (`human_in_loop: false` with non-empty `escalation_triggers`). Teaches: *business invariants are not JSON’s job*.
2. **Second cell** — Same dict shape **rejected** by the wrong top-level model (`WorkflowSpec` vs scaffold), then **accepted** by a **`TypeAdapter` union**. Teaches: *routing and validation are related*—you must validate against the **intended** contract.


In [ ]:
bad_but_valid_json = json.dumps(
    {
        "system_goal": "assistant",
        "capabilities": ["answer policy questions"],
        "human_in_loop": False,
        "escalation_triggers": [
            {"condition": "legal risk", "target_team": "legal", "severity": "high"}
        ],
    }
)
print("JSON parses:", json.loads(bad_but_valid_json))
try:
    WorkflowSpec.model_validate_json(bad_but_valid_json)
except ValidationError as e:
    print("\nPydantic rejects (model / business rules):\n", e)


JSON parses: {'system_goal': 'assistant', 'capabilities': ['answer policy questions'], 'human_in_loop': False, 'escalation_triggers': [{'condition': 'legal risk', 'target_team': 'legal', 'severity': 'high'}]}

Pydantic rejects (model / business rules):
 1 validation error for WorkflowSpec
  Value error, escalation_triggers is non-empty but human_in_loop is False [type=value_error, input_value={'system_goal': 'assistan...', 'severity': 'high'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


In [ ]:
# Union adapter: validate discriminated payloads from an API queue
WorkflowOrScaffold = TypeAdapter(WorkflowSpec | CodingScaffold)

# Wrong model for the payload → ValidationError even if JSON is fine
scaffold_like = {
    "language": "python",
    "input_type": "csv",
    "processing_steps": [{"step_id": "read", "description": "read file"}],
    "output_type": "report",
}
try:
    WorkflowSpec.model_validate(scaffold_like)
except ValidationError as e:
    print("WorkflowSpec cannot validate a scaffold-shaped dict:\n", e)

ok = WorkflowOrScaffold.validate_python(scaffold_like)
print("Same dict via union TypeAdapter ->", type(ok).__name__, ok.suggested_entrypoint)


WorkflowSpec cannot validate a scaffold-shaped dict:
 3 validation errors for WorkflowSpec
system_goal
  Field required [type=missing, input_value={'language': 'python', 'i...'output_type': 'report'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
capabilities
  Field required [type=missing, input_value={'language': 'python', 'i...'output_type': 'report'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
human_in_loop
  Field required [type=missing, input_value={'language': 'python', 'i...'output_type': 'report'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
Same dict via union TypeAdapter -> CodingScaffold main.py::main


## 4) Extraction prompts (LLM → structured objects)

**What this section does:** Uses `ChatOpenAI(...).with_structured_output(...)` **twice**—once for `WorkflowSpec`, once for `CodingScaffold`—with small system prompts to fill each model from **messy** `MESSY_WORKFLOW` / `MESSY_SCAFFOLD` text.

**Why it matters:** This is the fastest way to iterate on schemas in a notebook. It is **not** where trust should end.

### Why we still serialize and validate again in section 5

Extraction uses `with_structured_output`, then the graph **serializes** (`model_dump_json`) and **parses + validates again** (`model_validate_json`). Conceptually that is the point: **even a model that already emitted schema-shaped output can be wrong *after* it leaves the process**—or you may deserialize in another language, version, or service.

**We intentionally re-serialize and validate again because real systems cross process or service boundaries. Trust does not transfer just because an earlier node used a schema-aware extraction method.**

Without saying this out loud, learners often think: *“Why validate twice if `with_structured_output` already gave me a model?”* The answer is: **the in-memory object in node A is not what node B receives** once you cross a queue, HTTP, or another team’s deploy. Only the **contract + validation** at each boundary makes the handoff safe.

**This notebook** simulates that boundary by passing **`draft_json` as a string** between LangGraph nodes.


In [ ]:
WORKFLOW_EXTRACT_SYSTEM = """You extract a WorkflowSpec from messy stakeholder text.

Rules:
- capabilities: concrete verbs (policy_qa, document_summary, ticket_creation, ...).
- restricted_actions: things the bot must NOT do.
- escalation_triggers: each needs condition + target_team + severity.
- required_tools: integrations (vector_db, slack_post, crm_lookup) with purpose.
- human_in_loop must be true if any human escalation or approval is described.
- open_questions: unknowns; do not invent org details.
"""

SCAFFOLD_EXTRACT_SYSTEM = """You extract a CodingScaffold from a messy coding request.

Rules:
- processing_steps: ordered pipeline; step_id is snake_case; description is actionable.
- dependencies: packages or services if implied else empty.
- edge_cases: data quality, scale, failure modes mentioned or clearly implied.
- open_questions: ambiguities only.
"""

MESSY_WORKFLOW = (
    "We need an internal assistant that can answer policy questions, summarize uploaded documents, "
    "and escalate anything risky to legal. It should never execute financial transactions or send "
    "external email without a human approving first."
)

MESSY_SCAFFOLD = (
    "Can you make a small Python script that reads a CSV of orders, filters failed payments, "
    "dedupes by customer id, and writes a summary markdown report? Dates might be messy."
)


def extract_workflow_spec(text: str) -> WorkflowSpec:
    chain = llm.with_structured_output(WorkflowSpec)
    return chain.invoke(
        [SystemMessage(content=WORKFLOW_EXTRACT_SYSTEM), HumanMessage(content=text)]
    )


def extract_coding_scaffold(text: str) -> CodingScaffold:
    chain = llm.with_structured_output(CodingScaffold)
    return chain.invoke(
        [SystemMessage(content=SCAFFOLD_EXTRACT_SYSTEM), HumanMessage(content=text)]
    )


wf = extract_workflow_spec(MESSY_WORKFLOW)
cs = extract_coding_scaffold(MESSY_SCAFFOLD)
print("WorkflowSpec OK:", wf.system_goal[:60], "...")
print("CodingScaffold OK:", cs.suggested_entrypoint, len(cs.processing_steps), "steps")


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=WorkflowSpec(system_goal=...?'], escalation_count=1), input_type=WorkflowSpec])
  return self.__pydantic_serializer__.to_python(


WorkflowSpec OK: Internal assistant for policy Q&A, document summarization, a ...
CodingScaffold OK: main.py::main 7 steps


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CodingScaffold(language='...rypoint='main.py::main'), input_type=CodingScaffold])
  return self.__pydantic_serializer__.to_python(


## 5) Minimal LangGraph: classify → extract → **validate handoff** → artifact

**What this section does:** Builds a tiny graph whose **real lesson is the middle edge**: `draft_json` is treated as **untrusted wire data** and must pass `model_validate_json` before any “emit artifact” node runs.

**Why it matters:** In production, the failure mode is not “the LLM hallucinated”—it is “**something between two services changed the JSON**” (version skew, partial migration, bad default in an API gateway). Putting **validate** in the graph makes that failure **visible and testable**.

**Nodes (each is a boundary)**

| Node | Role |
|------|------|
| **`classify`** | Chooses `workflow` vs `coding` so the right extractor runs. |
| **`extract_*`** | Calls the LLM with structured output, then **drops JSON on the wire** (`draft_json`). |
| **`validate_*`** | **Mandatory** `model_validate_json`. Success → typed dict in state; failure → `last_error` and route to `validation_fail`. |
| **`emit_*`** | Turns **already validated** dicts into markdown briefs (boring string formatting—safe because validation passed). |
| **`validation_fail`** | Surfaces `ValidationError` text—stub for retry / dead-letter / human queue in real systems. |

**Trace** — The `trace` list is only for teaching: show the class the exact path through nodes.


In [ ]:
class RequestKind(BaseModel):
    kind: Literal["workflow", "coding"] = Field(
        description="workflow = agent/policy assistant spec; coding = script or service scaffold"
    )
    confidence: Literal["high", "medium", "low"] = "high"
    rationale: str = Field(max_length=200, default="")


class GraphState(TypedDict, total=False):
    user_message: str
    kind: str
    draft_json: str
    validated_workflow: dict[str, Any] | None
    validated_scaffold: dict[str, Any] | None
    last_error: str | None
    artifact: str
    trace: list[str]


def append_trace(state: GraphState, msg: str) -> list[str]:
    return [*state.get("trace", []), msg]


ROUTER_SYSTEM = '''Classify the user message as:
- workflow: building an assistant/agent with policies, tools, escalation, humans in the loop
- coding: building a script, CLI, ETL, report, or service scaffold

Return kind + brief rationale.'''


def node_classify(state: GraphState) -> GraphState:
    chain = llm.with_structured_output(RequestKind)
    rk = chain.invoke(
        [SystemMessage(content=ROUTER_SYSTEM), HumanMessage(content=state["user_message"])]
    )
    return {
        "kind": rk.kind,
        "trace": append_trace(state, f"classify -> {rk.kind} ({rk.confidence})"),
    }


def route_by_kind(state: GraphState) -> str:
    return state.get("kind") or "workflow"


def node_extract_workflow(state: GraphState) -> GraphState:
    spec = extract_workflow_spec(state["user_message"])
    # Simulate wire handoff: only JSON crosses the boundary
    raw = spec.model_dump_json()
    return {
        "draft_json": raw,
        "trace": append_trace(state, "extract_workflow -> draft_json set"),
    }


def node_extract_scaffold(state: GraphState) -> GraphState:
    spec = extract_coding_scaffold(state["user_message"])
    raw = spec.model_dump_json()
    return {
        "draft_json": raw,
        "trace": append_trace(state, "extract_scaffold -> draft_json set"),
    }


def node_validate_workflow(state: GraphState) -> GraphState:
    try:
        obj = WorkflowSpec.model_validate_json(state["draft_json"])
    except ValidationError as e:
        return {
            "validated_workflow": None,
            "last_error": str(e),
            "trace": append_trace(state, "validate_workflow FAILED"),
        }
    return {
        "validated_workflow": obj.model_dump(mode="json"),
        "last_error": None,
        "trace": append_trace(state, "validate_workflow OK"),
    }


def node_validate_scaffold(state: GraphState) -> GraphState:
    try:
        obj = CodingScaffold.model_validate_json(state["draft_json"])
    except ValidationError as e:
        return {
            "validated_scaffold": None,
            "last_error": str(e),
            "trace": append_trace(state, "validate_scaffold FAILED"),
        }
    return {
        "validated_scaffold": obj.model_dump(mode="json"),
        "last_error": None,
        "trace": append_trace(state, "validate_scaffold OK"),
    }


def workflow_to_artifact(d: dict[str, Any]) -> str:
    lines = [
        "## Agent workflow brief",
        f"**Goal:** {d['system_goal']}",
        f"**HITL:** {d['human_in_loop']}",
        "",
        "### Capabilities",
        *[f"- {c}" for c in d["capabilities"]],
        "",
        "### Restricted",
        *[f"- {x}" for x in d.get("restricted_actions") or []],
        "",
        "### Escalations",
        *[f"- {e['condition']} → {e['target_team']} ({e['severity']})" for e in d.get("escalation_triggers") or []],
        "",
        "### Tools",
        *[f"- {t['name']}: {t['purpose']}" for t in d.get("required_tools") or []],
    ]
    if d.get("open_questions"):
        lines += ["", "### Open questions", *[f"- {q}" for q in d["open_questions"]]]
    return "\n".join(lines)


def scaffold_to_artifact(d: dict[str, Any]) -> str:
    lines = [
        "## Coding scaffold",
        f"**Language:** {d['language']}",
        f"**Suggested entry:** {d.get('suggested_entrypoint', 'n/a')}",
        f"**Input:** {d['input_type']} → **Output:** {d['output_type']}",
        "",
        "### Steps",
    ]
    for i, s in enumerate(d["processing_steps"], 1):
        lines.append(f"{i}. `{s['step_id']}` — {s['description']}")
    if d.get("dependencies"):
        lines += ["", "### Dependencies", *[f"- {p}" for p in d["dependencies"]]]
    if d.get("edge_cases"):
        lines += ["", "### Edge cases", *[f"- {e}" for e in d["edge_cases"]]]
    if d.get("open_questions"):
        lines += ["", "### Open questions", *[f"- {q}" for q in d["open_questions"]]]
    return "\n".join(lines)


def node_emit_workflow(state: GraphState) -> GraphState:
    d = state.get("validated_workflow")
    art = workflow_to_artifact(d) if d else f"ERROR: {state.get('last_error')}"
    return {"artifact": art, "trace": append_trace(state, "emit_workflow")}


def node_emit_scaffold(state: GraphState) -> GraphState:
    d = state.get("validated_scaffold")
    art = scaffold_to_artifact(d) if d else f"ERROR: {state.get('last_error')}"
    return {"artifact": art, "trace": append_trace(state, "emit_scaffold")}


def node_validation_fail(state: GraphState) -> GraphState:
    msg = state.get("last_error") or "unknown validation error"
    return {
        "artifact": f"## Validation failed at handoff\n\n```\n{msg}\n```",
        "trace": append_trace(state, "validation_fail"),
    }


def route_after_validate_workflow(state: GraphState) -> str:
    return "ok" if state.get("validated_workflow") else "fail"


def route_after_validate_scaffold(state: GraphState) -> str:
    return "ok" if state.get("validated_scaffold") else "fail"


def build_app():
    g = StateGraph(GraphState)
    g.add_node("classify", node_classify)
    g.add_node("extract_workflow", node_extract_workflow)
    g.add_node("extract_scaffold", node_extract_scaffold)
    g.add_node("validate_workflow", node_validate_workflow)
    g.add_node("validate_scaffold", node_validate_scaffold)
    g.add_node("emit_workflow", node_emit_workflow)
    g.add_node("emit_scaffold", node_emit_scaffold)
    g.add_node("validation_fail", node_validation_fail)

    g.add_edge(START, "classify")
    g.add_conditional_edges(
        "classify",
        route_by_kind,
        {"workflow": "extract_workflow", "coding": "extract_scaffold"},
    )
    g.add_edge("extract_workflow", "validate_workflow")
    g.add_edge("extract_scaffold", "validate_scaffold")
    g.add_conditional_edges(
        "validate_workflow",
        route_after_validate_workflow,
        {"ok": "emit_workflow", "fail": "validation_fail"},
    )
    g.add_conditional_edges(
        "validate_scaffold",
        route_after_validate_scaffold,
        {"ok": "emit_scaffold", "fail": "validation_fail"},
    )
    g.add_edge("emit_workflow", END)
    g.add_edge("emit_scaffold", END)
    g.add_edge("validation_fail", END)
    return g.compile()


app = build_app()


In [ ]:
# End-to-end: workflow message
out_wf = app.invoke({"user_message": MESSY_WORKFLOW, "trace": []})
print(out_wf["artifact"][:1200])
print("\n--- trace:", " -> ".join(out_wf.get("trace", [])))


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RequestKind(kind='workflo... constructing an agent'), input_type=RequestKind])
  return self.__pydantic_serializer__.to_python(


## Agent workflow brief
**Goal:** Internal assistant for policy support and document handling
**HITL:** True

### Capabilities
- policy_qa
- Answer employee questions about internal policies
- document_summary
- Summarize uploaded documents (PDF, DOCX, etc.)

### Restricted
- financial_transaction_execution
- Never perform any monetary transfers or financial operations
- external_email_sending_without_human_approval
- Do not send external email unless a human explicitly approves

### Escalations
- risky_content_detected → legal (high)

### Tools
- vector_db: Store and retrieve policy documents for fast semantic search during policy Q&A
- doc_parser: Extract text from uploaded files (PDF, DOCX, etc.) for summarization

### Open questions
- What criteria define "risky content" that should trigger escalation to legal?
- Which specific legal team or contact should receive escalations?
- What sources (policy manuals, guidelines) should be ingested into the vector_db?
- What format(s) of doc

In [ ]:
# End-to-end: coding message
out_cs = app.invoke({"user_message": MESSY_SCAFFOLD, "trace": []})
print(out_cs["artifact"][:1200])
print("\n--- trace:", " -> ".join(out_cs.get("trace", [])))


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RequestKind(kind='coding'...is a request for code.'), input_type=RequestKind])
  return self.__pydantic_serializer__.to_python(


## Coding scaffold
**Language:** python
**Suggested entry:** main.py::main
**Input:** csv → **Output:** markdown

### Steps
1. `load_csv` — Read the orders CSV into a DataFrame (or list of dicts) using pandas, allowing configurable delimiter, encoding, and handling of malformed lines.
2. `normalize_dates` — Parse the order date column with dateutil (or pandas.to_datetime) using `errors='coerce'` to turn unparseable values into NaT, then optionally fill or drop those rows.
3. `filter_failed_payments` — Identify the column that indicates payment status (e.g., `payment_status` or `payment_result`) and keep only rows where the status denotes a successful payment (e.g., values like 'paid', 'success', 1).
4. `dedupe_by_customer` — Group the filtered data by `customer_id` and deduplicate. Decide a rule (e.g., keep the most recent order by normalized date, or the first/last occurrence) and apply it with `drop_duplicates` or `groupby` + `agg`.
5. `generate_summary` — Compute summary statistics 

# Force a handoff validation failure (coding path)

**What this section does:** Takes a **successful** `draft_json` from a real graph run, then **mutates a nested field**—the first `processing_steps[].step_id` before validation.

**Why it matters:** Production bugs are rarely “delete `processing_steps` entirely.” They look like **one bad field** from a UI, a spreadsheet import, or a “helpful” formatter.

**Note on `"Read CSV"`:** A human-friendly label like `"Read CSV"` often becomes `read_csv` after **normalization** in our `field_validator` (spaces → underscores, lowercased)—so it might **pass** slug checks. The code below uses a **`step_id` that still violates the slug regex after normalization** (illegal character), so `ValidationError` fires on the **nested** rule, not only on a missing required key.

If you want to discuss Title Case labels that **do** normalize to valid slugs, use that as a second talking point: **normalization is a policy choice** some teams prefer to **fail closed** on any non-slug input so bad data never silently becomes canonical.


In [ ]:
corrupted_state = app.invoke({"user_message": MESSY_SCAFFOLD, "trace": []})
raw = corrupted_state.get("draft_json") or ""
parsed = json.loads(raw)
steps = parsed.get("processing_steps") or []
if not steps:
    raise RuntimeError("Expected processing_steps from extraction; rerun the LangGraph cells above.")

# Nested corruption: mimics a UI / export / microservice "helpfully" rewriting one label.
# "Read CSV" often normalizes to read_csv and would PASS our slug validator — so we inject
# a character the slug regex rejects (still a realistic partial-sanitizer bug).
parsed["processing_steps"][0]["step_id"] = "read@csv"

bad_json = json.dumps(parsed)

broken = {
    "user_message": MESSY_SCAFFOLD,
    "kind": "coding",
    "draft_json": bad_json,
    "trace": ["manual_inject_corrupt_handoff_nested_step_id"],
}
v = node_validate_scaffold(broken)
emit = node_emit_scaffold({**broken, **v})
print(emit["artifact"][:1200])


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RequestKind(kind='coding'...a small piece of code.'), input_type=RequestKind])
  return self.__pydantic_serializer__.to_python(


ERROR: 1 validation error for CodingScaffold
processing_steps.0.step_id
  Value error, step_id must match slug pattern, got 'read@csv' [type=value_error, input_value='read@csv', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
